In [ ]:
import cell2cell as c2c

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#working env
import os
#os.chdir(r'\\isdsyn1-1.isd.med.uni-muenchen.de\BD-Bernhagen\Simon Ebert\snRNAseq_Python_env_normalMT')
os.chdir(r'W://Simon Ebert\snRNAseq_Python_env_normalMT')
os.listdir()

In [ ]:
data_folder = '//isdsyn1-1.isd.med.uni-muenchen.de/BD-Bernhagen/Simon Ebert/snRNAseq_Python_env_normalMT/cell2cell_tensor/output/quickstart/'
output_folder = '//isdsyn1-1.isd.med.uni-muenchen.de/BD-Bernhagen/Simon Ebert/snRNAseq_Python_env_normalMT/cell2cell_tensor/output/'
c2c.io.directories.create_directory(data_folder)
c2c.io.directories.create_directory(output_folder)

In [ ]:
factors = c2c.io.load_tensor_factors(output_folder + '/Loadings.xlsx')

In [ ]:
from collections import defaultdict

element_dict = defaultdict(lambda: 'Unknown')

In [ ]:
context_dict = element_dict.copy()

context_dict.update({'HFD' : 'HFD+vehicle',
                     'HFD_L1' : 'HFD+msR4M-L1',
              
                    })

In [ ]:
from matplotlib.pyplot import rc_context
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
groups_order = ["HFD+vehicle", "HFD+msR4M-L1"]
fig_filename = output_folder + '/BALF-Severity-Boxplots.pdf'
with rc_context({"figure.figsize": (4, 3)}):
    _ = c2c.plotting.context_boxplot(context_loadings=factors['Contexts'],
                                    metadict=context_dict,
                                    nrows=3,
                                    figsize=(10, 10),
                                    group_order=groups_order,
                                    statistical_test='t-test_ind',
                                    pval_correction='fdr_bh',
                                    cmap='plasma',
                                    verbose=False, 
                                    filename=fig_filename
                                    )

In [ ]:
def severity_rank(x):
    if x == 'HFD+vehicle':
        ranking = 1
    elif x == 'HFD+msR4M-L1':
        ranking = 2
    return ranking

In [ ]:
sev_rank = pd.DataFrame(index=factors['Contexts'].index)
sev_rank['Severity-Ranking'] = [severity_rank(context_dict[idx]) for idx in factors['Contexts'].index]
sev_rank.head()

In [ ]:
nfactors = factors['Contexts'].shape[1]

In [ ]:
import scipy
for i in range(1, nfactors+1):
    factor = 'Factor {}'.format(i)
    print(factor, scipy.stats.spearmanr(factors['Contexts'][factor],
                                sev_rank['Severity-Ranking']))

In [ ]:
# Generate color by COVID-19 severity for each sample
condition_colors = c2c.plotting.aesthetics.get_colors_from_labels(["HFD+vehicle", "HFD+msR4M-L1"],
                                                                  cmap='plasma')

# Map these colors to each sample name
color_dict = {k : condition_colors[v] for k, v in context_dict.items()}

# Generate a dataframe used as input for the clustermap
col_colors = pd.Series(color_dict)
col_colors = col_colors.to_frame()
col_colors.columns = ['Condition']

In [ ]:
col_colors.head()

In [ ]:
sample_cm = c2c.plotting.loading_clustermap(factors['Contexts'],
                                            use_zscore=False, # Whether standardizing the loadings across factors
                                            col_colors=col_colors, # Change this to color by other properties
                                            figsize=(6, 4),
                                            dendrogram_ratio=0.3,
                                            cbar_fontsize=12,
                                            tick_fontsize=12,
                                            filename=output_folder + 'Clustermap-Contexts.pdf'
                                           )

#Insert legend
plt.sca(sample_cm.ax_heatmap)
legend = c2c.plotting.aesthetics.generate_legend(color_dict=condition_colors,
                                                 bbox_to_anchor=(1.25, 0.9), # Position of the legend (X, Y)
                                                 title='Condition',
                                                 fontsize=12,
                                                 
                                                )

In [ ]:
lr_cm = c2c.plotting.loading_clustermap(factors['Ligand-Receptor Pairs'],
                                        loading_threshold=0.1, # To consider only top LRs
                                        use_zscore=False, # Whether standardizing the loadings across factors
                                        figsize=(15, 8),
                                        filename=output_folder + 'Clustermap-LRs.pdf',
                                        tick_fontsize=8,
                                        row_cluster=False # To avoid clustering of factors
                                       )

In [ ]:
selected_factor = 'Factor 4'

In [ ]:
loading_product = c2c.analysis.tensor_downstream.get_joint_loadings(factors,
                                                                    dim1='Sender Cells',
                                                                    dim2='Receiver Cells',
                                                                    factor=selected_factor,
                                                                   )

In [ ]:
lprod_cm = c2c.plotting.loading_clustermap(loading_product.T, # Remove .T to transpose the axes
                                           use_zscore=False, # Whether standardizing the loadings across factors
                                           figsize=(4, 6),
                                           filename=output_folder + 'Clustermap-CC-Pairs.pdf',
                                           cbar_label='Loading Product',

                                          )

In [ ]:
lr_cell_product = c2c.analysis.tensor_downstream.get_lr_by_cell_pairs(factors,
                                                                      lr_label='Ligand-Receptor Pairs',
                                                                      sender_label='Sender Cells',
                                                                      receiver_label='Receiver Cells',
                                                                      order_cells_by='receivers',
                                                                      factor=selected_factor,
                                                                      cci_threshold=None, # None for considering all cell-cell pairs.
                                                                      lr_threshold=0.05 # Prioritize important pairs. None for considering all of them
                                                                     )

In [ ]:
lsr_prod_cm = c2c.plotting.loading_clustermap(lr_cell_product,
                                              use_zscore=False, # Whether standardizing the loadings across factors
                                              figsize=(18, 12),
                                              filename=output_folder + 'Clustermap-LR-CC-Pairs.pdf',
                                              cbar_label='Loading Product',
                                              yticklabels=1,
                                              tick_fontsize=8,
                                             )

In [ ]:
# Get all outer products as adjacency matrices, one per factor
networks = c2c.analysis.tensor_downstream.get_factor_specific_ccc_networks(factors,
                                                                           sender_label='Sender Cells',
                                                                           receiver_label='Receiver Cells',
                                                                           )

In [ ]:
# Then, flatten the adjacency matrices
network_by_factors = c2c.analysis.tensor_downstream.flatten_factor_ccc_networks(networks, orderby='receivers')

# And we can plot the distributions of the weights for each factor-specific network
_ = plt.hist(network_by_factors.values.flatten(), bins = 50)

In [ ]:
threshold = 0.05

In [ ]:

c2c.plotting.ccc_networks_plot(factors,
                               included_factors=["Factor 4"],
                               ccc_threshold=threshold, # Only important communication
                               nrows=1,
                               panel_size=(6, 6), # This changes the size of each figure panel.
                               filename=output_folder + 'Factor-Networks.pdf',
                               factor_title_size=14,
                               node_size=200,
                               edge_width=10,
                               node_label_size=12,
                               network_layout='spring', # 'spring' or 'circular'
                              

                              )

In [ ]:
c2c.analysis.tensor_downstream.compute_gini_coefficients(factors,
                                                         sender_label='Sender Cells',
                                                         receiver_label='Receiver Cells'
                                                        ).sort_values('Gini')